# Da CSV a SQLite (normalizzando i dati)

Partiamo dal file delle pubblicazioni ottenuto da OpenData Lombardia, che contiene informazioni sulle pubblicazioni scientifiche prodotte da ricercatori affiliati a istituzioni lombarde.

Iniziamo a leggere i dati dal file CSV e a esplorarli.

In [1]:
import csv

Raccoglimo gli `header` e il resto delle righe in `data`.

In [2]:
import gzip

data = []
with gzip.open('pubblicazioni.csv.gz', 'rt', encoding='utf-8') as inf:
  reader = csv.reader(inf)
  header = next(reader)
  for row in reader:
    data.append(row)

In [3]:
for i, h in enumerate(header):
  print(f'{i}: {h}')

0: HANDLE
1: TITOLO
2: AUTORI_INTERNI
3: TIPOLOGIA
4: ANNO
5: ABSTRACT
6: LINGUA
7: CITAZIONE
8: FILE_NOME
9: FILE_FORMATO
10: FILE_VERSIONE
11: FILE_ACCESSO
12: FILE_URL
13: LICENZA


Il campo `AUTORI_INTERNI` contiene gli autori divisi da `;` con nome e cognome divisi da `,`.

In [4]:
nomi_e_cognomi = set() # usiamo un set per evitare duplicati
for row in data:
  if not row[2]: continue
  for cognome_nome in row[2].split(';'):
    cognome, nome = cognome_nome.split(',')
    nomi_e_cognomi.add((nome.strip(), cognome.strip()))

len(nomi_e_cognomi)

15315

Costriuiamo una mappa da nome e cognome a un ID univoco, che ci servirà per normalizzare i dati e costruire le tabelle relazionali.

In [5]:
nomi_e_cognomi2id = dict()

for id_autore, (nome, cognome) in enumerate(nomi_e_cognomi):
  nomi_e_cognomi2id[(nome, cognome)] = id_autore

Costruiamo ora un elenco di coppie contenenti il numero ordinare di ciascuna pubblicazione e gli ID di ciascuno degli autori interni che vi figurano.

In [6]:
id_pubblicazione_e_id_autore = []
for id_pubblicazione, row in enumerate(data):
  if not row[2]: continue
  for cognome_nome in row[2].split(';'):
    cognome, nome = cognome_nome.split(',')
    id_autore = nomi_e_cognomi2id[(nome.strip(), cognome.strip())]
    id_pubblicazione_e_id_autore.append((id_pubblicazione, id_autore))

len(id_pubblicazione_e_id_autore)

176251

Ora siamo pronti a costruire le tabelle del DB, raccogliendo per ciascuna i suoi campi.

Iniziamo da quella delle pubblicazioni, per cui ci aiutiamo con gli `header` (saltiamo gli `AUTORI_INTERNI` che sostituiremo con le coppie di identificatori calcolate prima).

In [7]:
table2fields = dict() 

table2fields['pubblicazioni'] = [f'{h} TEXT' for h in header if h != 'AUTORI_INTERNI']

In [8]:
table2fields['autori'] = ['NOME TEXT', 'COGNOME TEXT']
table2fields['pubblicazioni_autori'] = ['ID_PUBBLICAZIONE INTEGER', 'ID_AUTORE INTEGER']

A ciascuna tabella aggiungiamo un campo `ID` che sarà la chiave primaria.

In [9]:
for table, fields in table2fields.items():
  fields.insert(0, 'ID INTEGER PRIMARY KEY')

Siamo pronti a genreare le tabelle.

In [10]:
import sqlite3

conn = sqlite3.connect('pubblicazioni.db')

for table, fields in table2fields.items():
  SQL = f"DROP TABLE IF EXISTS {table}; CREATE TABLE {table} (\n  " + ',\n'.join(fields) + "\n);"
  with conn: conn.executescript(SQL)

Ora popoliamo le tabelle coi dati

In [11]:
data_no_autori_interni = []
for id_pubblicazione, row in enumerate(data):
  data_no_autori_interni.append(
    [id_pubblicazione] + [row[i] for i, h in enumerate(header) if h != 'AUTORI_INTERNI']  
  )

with conn:
  conn.executemany("INSERT INTO pubblicazioni (ID, " + ','.join(h for h in header if h != 'AUTORI_INTERNI') + ") VALUES (" + ','.join('?' for _ in header) + ")", data_no_autori_interni)

In [12]:
with conn:
  conn.executemany("INSERT INTO autori (ID, NOME, COGNOME) VALUES (?, ?, ?)", [(id_autore, nome, cognome) for (nome, cognome), id_autore in nomi_e_cognomi2id.items()])

In [13]:
with conn:
  conn.executemany("INSERT INTO pubblicazioni_autori (ID, ID_PUBBLICAZIONE, ID_AUTORE) VALUES (?, ?, ?)", [(id_coppia, id_pubblicazione, id_autore) for id_coppia, (id_pubblicazione, id_autore) in enumerate(id_pubblicazione_e_id_autore)])

In [14]:
conn.row_factory = sqlite3.Row
with conn:
  for row in conn.execute("""
    SELECT * FROM pubblicazioni, autori, pubblicazioni_autori WHERE 
      pubblicazioni.ID = pubblicazioni_autori.ID_PUBBLICAZIONE AND 
      autori.ID = pubblicazioni_autori.ID_AUTORE AND 
      autori.COGNOME = 'SANTINI' and autori.NOME = 'MASSIMO'
    """):
    for k, v in dict(row).items():
      print(f'{k}: {v}')
    print('=' * 80)

ID: 8039
HANDLE: https://hdl.handle.net/2434/1116789
TITOLO: The Molecules Gateway: A Homogeneous, Searchable Database of 150k Annotated Molecules from Actinomycetes
TIPOLOGIA: 01 - Articolo su periodico
ANNO: 2024
ABSTRACT: Natural products are a sustainable resource for drug discovery, but their identification in complex mixtures remains a daunting task. We present an automated pipeline that compares, harmonizes and ranks the annotations of LC-HRMS data by different tools. When applied to 7,400 extracts derived from 6,566 strains belonging to 86 actinomycete genera, it yielded 150,000 molecules after processing over 50 million MS features. The web-based Molecules Gateway provides a highly interactive access to experimental and calculated data for these molecules, along with the metadata related to extracts and producer strains. We show how the Molecules Gateway can be used to rapidly identify known hard to find microbial products, unreported analogs of known families and not yet desc